# Export CNN1D to TFLite for Raspberry Pi

Convert PyTorch trained model to TFLite format for Pi 4 runtime deployment.

Workflow:
1. Load PyTorch model checkpoint
2. Rebuild model in Keras + transfer weights (bỏ qua ONNX — tránh lỗi dtype Conv1d)
3. Test inference với ai-edge-litert (cùng package với runtime)
4. Đo kích thước và tốc độ inference

## 0. Setup

In [1]:
import sys
from pathlib import Path
import numpy as np
import torch
import torch.nn as nn

PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT))

print("Checking dependencies...")
try:
    import tensorflow as tf
    print(f"  tensorflow:     OK (version {tf.__version__})")
except ImportError:
    print("  tensorflow:     NOT INSTALLED — pip install tensorflow")

try:
    from ai_edge_litert.interpreter import Interpreter as _AELInterp
    print("  ai-edge-litert: OK")
except ImportError:
    print("  ai-edge-litert: NOT INSTALLED — pip install ai-edge-litert")

# Model paths
MODEL_DIR    = PROJECT_ROOT / "models"
PT_MODEL     = MODEL_DIR / "soc_cnn1d.pt"
TFLITE_MODEL = MODEL_DIR / "soc_cnn1d.tflite"

print(f"\nModel paths:")
print(f"  PyTorch: {PT_MODEL}")
print(f"  TFLite:  {TFLITE_MODEL}")

Checking dependencies...
  tensorflow:     OK (version 2.21.0)
  ai-edge-litert: OK

Model paths:
  PyTorch: D:\DoAn\SourceCode\models\soc_cnn1d.pt
  TFLite:  D:\DoAn\SourceCode\models\soc_cnn1d.tflite


## 1. Load PyTorch Model

In [2]:
class CNN1D(nn.Module):
    def __init__(self, in_channels=4):
        super(CNN1D, self).__init__()
        self.conv1 = nn.Sequential(
            nn.Conv1d(in_channels, 64, kernel_size=3, padding=1),
            nn.BatchNorm1d(64),
            nn.ReLU()
        )
        self.conv2 = nn.Sequential(
            nn.Conv1d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm1d(128),
            nn.ReLU()
        )
        self.pool = nn.AdaptiveAvgPool1d(1)
        self.fc = nn.Sequential(
            nn.Dropout(0.3),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(64, 1)
        )

    def forward(self, x):
        x = x.permute(0, 2, 1)
        x = self.conv1(x)
        x = self.conv2(x)
        x = self.pool(x).squeeze(-1)
        return self.fc(x)

# Load checkpoint
# weights_only=False vì checkpoint lưu numpy scalars (feature_norm, best_mae)
# File được tạo bởi notebook training trong cùng project — nguồn tin cậy.
checkpoint = torch.load(str(PT_MODEL), map_location='cpu', weights_only=False)
model = CNN1D(in_channels=4)
model.load_state_dict(checkpoint['model_state'])
model.eval()

print(f"Loaded model from {PT_MODEL}")
print(f"Best MAE during training: {checkpoint['best_mae']:.6f}")

Loaded model from D:\DoAn\SourceCode\models\soc_cnn1d.pt
Best MAE during training: 0.060951


In [3]:
import yaml

# Đọc feature_norm và label_scale từ checkpoint, ghi vào configs/model.yaml
feature_norm = checkpoint['feature_norm']
label_scale  = float(checkpoint['label_scale'])

config_path = PROJECT_ROOT / "configs" / "model.yaml"
with open(config_path, encoding="utf-8") as f:
    cfg = yaml.safe_load(f)

cfg['normalization'] = feature_norm   # ghi đè section sẵn có (thay số hardcode bằng giá trị thật)
cfg['label_scale']   = label_scale    # thêm key mới

with open(config_path, "w", encoding="utf-8") as f:
    yaml.dump(cfg, f, allow_unicode=True, default_flow_style=False)

print(f"Đã cập nhật {config_path}")
print(f"normalization (từ training data):")
for col, v in feature_norm.items():
    print(f"  {col}: min={v['min']:.4f}  max={v['max']:.4f}")
print(f"label_scale: {label_scale}")

Đã cập nhật D:\DoAn\SourceCode\configs\model.yaml
normalization (từ training data):
  pack_voltage_v: min=69.8820  max=73.8180
  pack_current_a: min=-2.2600  max=21.2200
  temp_c: min=0.0000  max=46.0000
  speed_kmh: min=0.0000  max=54.1000
label_scale: 100.0


## 2. Convert to TFLite (Keras + weight transfer)

In [4]:
import tensorflow as tf

# -----------------------------------------------------------------------
# Bước 1 — Xây Keras model tương đương
# Input (1, 60, 4) channels-last — Conv1D Keras dùng NLC nên không cần permute
# epsilon=1e-5 khớp với PyTorch BatchNorm1d mặc định
# -----------------------------------------------------------------------
inp = tf.keras.Input(shape=(60, 4), dtype=tf.float32, name="input")

x = tf.keras.layers.Conv1D(64, 3, padding="same", name="conv1_conv")(inp)
x = tf.keras.layers.BatchNormalization(epsilon=1e-5, name="conv1_bn")(x)
x = tf.keras.layers.ReLU(name="conv1_relu")(x)

x = tf.keras.layers.Conv1D(128, 3, padding="same", name="conv2_conv")(x)
x = tf.keras.layers.BatchNormalization(epsilon=1e-5, name="conv2_bn")(x)
x = tf.keras.layers.ReLU(name="conv2_relu")(x)

x = tf.keras.layers.GlobalAveragePooling1D(name="pool")(x)
x = tf.keras.layers.Dropout(0.3)(x)
x = tf.keras.layers.Dense(64, activation="relu", name="fc1")(x)
x = tf.keras.layers.Dropout(0.2)(x)
x = tf.keras.layers.Dense(1, name="fc2")(x)

keras_model = tf.keras.Model(inputs=inp, outputs=x)

# -----------------------------------------------------------------------
# Bước 2 — Chuyển weights PyTorch → Keras
# Conv1d  (out, in, k) → Keras Conv1D (k, in, out)  : transpose(2,1,0)
# Linear  (out, in)    → Keras Dense  (in, out)      : .T
# BatchNorm: gamma, beta, running_mean, running_var   : thứ tự Keras set_weights
# -----------------------------------------------------------------------
sd = model.state_dict()

keras_model.get_layer("conv1_conv").set_weights([
    sd["conv1.0.weight"].numpy().transpose(2, 1, 0),
    sd["conv1.0.bias"].numpy(),
])
keras_model.get_layer("conv1_bn").set_weights([
    sd["conv1.1.weight"].numpy(),
    sd["conv1.1.bias"].numpy(),
    sd["conv1.1.running_mean"].numpy(),
    sd["conv1.1.running_var"].numpy(),
])
keras_model.get_layer("conv2_conv").set_weights([
    sd["conv2.0.weight"].numpy().transpose(2, 1, 0),
    sd["conv2.0.bias"].numpy(),
])
keras_model.get_layer("conv2_bn").set_weights([
    sd["conv2.1.weight"].numpy(),
    sd["conv2.1.bias"].numpy(),
    sd["conv2.1.running_mean"].numpy(),
    sd["conv2.1.running_var"].numpy(),
])
keras_model.get_layer("fc1").set_weights([
    sd["fc.1.weight"].numpy().T,
    sd["fc.1.bias"].numpy(),
])
keras_model.get_layer("fc2").set_weights([
    sd["fc.4.weight"].numpy().T,
    sd["fc.4.bias"].numpy(),
])

# -----------------------------------------------------------------------
# Bước 3 — Sanity check: PyTorch ≈ Keras trên cùng input
# -----------------------------------------------------------------------
_x = np.random.randn(1, 60, 4).astype(np.float32)

with torch.no_grad():
    pt_out = float(model(torch.FloatTensor(_x)).squeeze())
keras_out = float(keras_model(_x, training=False).numpy().squeeze())

diff = abs(pt_out - keras_out)
print(f"PyTorch: {pt_out:.6f}   Keras: {keras_out:.6f}   diff: {diff:.2e}")
assert diff < 1e-3, f"Weight transfer thất bại (diff={diff:.4f}) — kiểm tra lại tên layer"
print("Weight transfer OK ✓")

# -----------------------------------------------------------------------
# Bước 4 — Keras → TFLite (standard path, không cần ONNX)
# -----------------------------------------------------------------------
converter = tf.lite.TFLiteConverter.from_keras_model(keras_model)
tflite_bytes = converter.convert()

with open(str(TFLITE_MODEL), "wb") as f:
    f.write(tflite_bytes)

print(f"\nExported to TFLite: {TFLITE_MODEL}")
print(f"File size: {TFLITE_MODEL.stat().st_size / 1024:.1f} KB")

PyTorch: 2.630556   Keras: 2.630556   diff: 4.77e-07
Weight transfer OK ✓
INFO:tensorflow:Assets written to: C:\Users\ADMIN\AppData\Local\Temp\tmpj62xae2x\assets


INFO:tensorflow:Assets written to: C:\Users\ADMIN\AppData\Local\Temp\tmpj62xae2x\assets


Saved artifact at 'C:\Users\ADMIN\AppData\Local\Temp\tmpj62xae2x'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 60, 4), dtype=tf.float32, name='input')
Output Type:
  TensorSpec(shape=(None, 1), dtype=tf.float32, name=None)
Captures:
  2114035677712: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2114037690640: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2114037689680: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2114037691984: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2114037689872: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2114037689488: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2114037691600: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2114037691792: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2114037691216: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2114037692944: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2114037692752: Ten

## 3. Test TFLite Inference

In [5]:
from ai_edge_litert.interpreter import Interpreter

# Dùng ai_edge_litert — cùng package với runtime src/soc_inference/inference.py
interpreter = Interpreter(model_path=str(TFLITE_MODEL))
interpreter.allocate_tensors()

input_details  = interpreter.get_input_details()
output_details = interpreter.get_output_details()

print(f"Input shape:  {input_details[0]['shape']}  dtype={input_details[0]['dtype'].__name__}")
print(f"Output shape: {output_details[0]['shape']}")

# Test input shape: (1, 60, 4) — khớp với sample_input khi export
test_input = np.random.randn(1, 60, 4).astype(np.float32)

interpreter.set_tensor(input_details[0]['index'], test_input)
interpreter.invoke()

tflite_output = interpreter.get_tensor(output_details[0]['index'])

# So sánh với PyTorch (kỳ vọng lệch < 1e-4 vì ai-edge-torch giữ fp32)
with torch.no_grad():
    pt_output = model(torch.FloatTensor(test_input)).numpy()

diff = np.abs(tflite_output - pt_output)
print(f"\nInference comparison (random input):")
print(f"  PyTorch output: {pt_output[0, 0]:.6f}")
print(f"  TFLite output:  {tflite_output[0, 0]:.6f}")
print(f"  Difference:     {diff[0, 0]:.6f}  (kỳ vọng < 1e-4)")

Input shape:  [ 1 60  4]  dtype=float32
Output shape: [1 1]

Inference comparison (random input):
  PyTorch output: 2.528841
  TFLite output:  2.528840
  Difference:     0.000000  (kỳ vọng < 1e-4)


## 5. Benchmarks

## 4. Benchmarks